# 🏆 Notebook 09: Final Model Comparison & Benchmarking
## Cross-Module Evaluation of All Supervised Models Across Performance Metrics

In [ ]:
import sys
import os
import time
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, confusion_matrix

from src.utils import load_numpy, load_model
from src.evaluation_metrics import compute_all_metrics

X_test = load_numpy(os.path.join(PROJECT_ROOT, "data", "processed", "X_test.npy"))
y_test_bin = load_numpy(os.path.join(PROJECT_ROOT, "data", "processed", "y_test_binary.npy"))
y_test_multi = load_numpy(os.path.join(PROJECT_ROOT, "data", "processed", "y_test_multi.npy"))

pca_model = load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "pca_model.pkl"))
lda_model = load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "lda_model.pkl"))
svd_model = load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "svd_model.pkl"))

X_test_pca = pca_model.transform(X_test)
X_test_lda = lda_model.transform(X_test)
X_test_svd = svd_model.transform(X_test)

In [ ]:
models_registry = {
    "Logistic Regression": (load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "logistic_regression.pkl")), X_test, "Mod 2"),
    "Decision Tree (Gini)": (load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "decision_tree_gini.pkl")), X_test, "Mod 2"),
    "Decision Tree (CART)": (load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "decision_tree_cart_classifier.pkl")), X_test, "Mod 2"),
    "Decision Stump": (load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "decision_stump.pkl")), X_test, "Mod 3"),
    "AdaBoost": (load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "adaboost.pkl")), X_test, "Mod 3"),
    "XGBoost": (load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "xgboost_model.pkl")), X_test, "Mod 3"),
    "Bagging": (load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "bagging_model.pkl")), X_test, "Mod 3"),
    "Subagging": (load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "subagging_model.pkl")), X_test, "Mod 3"),
    "Random Forest": (load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "random_forest.pkl")), X_test, "Mod 3"),
    "Voting Classifier (Hard)": (load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "voting_hard.pkl")), X_test, "Mod 3"),
    "Voting Classifier (Soft)": (load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "voting_soft.pkl")), X_test, "Mod 3"),
    "Stacking Classifier": (load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "stacking_model.pkl")), X_test, "Mod 3"),
    "SVM Linear": (load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "svm_linear.pkl")), X_test, "Mod 4"),
    "SVM RBF": (load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "svm_rbf.pkl")), X_test, "Mod 4"),
    "SVM Poly": (load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "svm_poly.pkl")), X_test, "Mod 4")
}

In [ ]:
records = []
roc_plot_data = {}

for name, (model, data_matrix, module_tag) in models_registry.items():
    t0 = time.time()
    preds = model.predict(data_matrix)
    infer_time = time.time() - t0
    
    probs = None
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(data_matrix)[:, 1]
    elif hasattr(model, "decision_function"):
        df_scores = model.decision_function(data_matrix)
        if df_scores.ndim == 1:
            probs = (df_scores - df_scores.min()) / (df_scores.max() - df_scores.min() + 1e-8)
            
    metrics = compute_all_metrics(y_test_bin, preds, probs)
    
    if probs is not None:
        fpr, tpr, _ = roc_curve(y_test_bin, probs)
        roc_plot_data[name] = (fpr, tpr, metrics["auc"])
        
    records.append({
        "Module": module_tag,
        "Model": name,
        "Accuracy": metrics["accuracy"],
        "Precision": metrics["precision"],
        "Recall": metrics["recall"],
        "F1-Score": metrics["f1"],
        "Kappa": metrics["kappa"],
        "Specificity": metrics["specificity"],
        "Sensitivity": metrics["sensitivity"],
        "AUC": metrics["auc"],
        "Inference Time (s)": infer_time
    })

comparison_df = pd.DataFrame(records)
comparison_df = comparison_df.sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
print(comparison_df.to_string())
comparison_df.to_csv(os.path.join(PROJECT_ROOT, "results", "model_comparison.csv"), index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
y_positions = np.arange(len(comparison_df))
bars = ax.barh(y_positions, comparison_df["F1-Score"], color="steelblue", edgecolor="black", height=0.6)
ax.set_yticks(y_positions)
ax.set_yticklabels(comparison_df["Model"])
ax.invert_yaxis()
ax.set_xlabel("F1-Score")
ax.set_title("Master Comparison: Models Ranked by F1-Score on NSL-KDD Test Set")
ax.set_xlim(0.0, 1.0)

for bar, score in zip(bars, comparison_df["F1-Score"]):
    ax.text(score + 0.01, bar.get_y() + bar.get_height() / 2, f"{score:.4f}", va="center", fontsize=9)

plt.tight_layout()
fig.savefig(os.path.join(PROJECT_ROOT, "plots", "final", "all_models_comparison.png"), dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

top_5_models = comparison_df["Model"].head(5).tolist()
for name in top_5_models:
    if name in roc_plot_data:
        fpr, tpr, auc_val = roc_plot_data[name]
        ax.plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.4f})", linewidth=2)

ax.plot([0, 1], [0, 1], color="grey", linestyle="--")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves: Top 5 Ranked Classifiers")
ax.legend(loc="lower right")
plt.tight_layout()
fig.savefig(os.path.join(PROJECT_ROOT, "plots", "final", "best_model_roc.png"), dpi=150)
plt.show()

In [ ]:
best_model_name = comparison_df.iloc[0]["Model"]
best_model = models_registry[best_model_name][0]
best_data = models_registry[best_model_name][1]

best_preds = best_model.predict(best_data)
cm = confusion_matrix(y_test_bin, best_preds)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=["Normal", "Attack"], yticklabels=["Normal", "Attack"], ax=ax)
ax.set_title(f"Confusion Matrix: Best Model ({best_model_name})")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
plt.tight_layout()
fig.savefig(os.path.join(PROJECT_ROOT, "plots", "final", "final_confusion_matrix.png"), dpi=150)
plt.show()